# KuchoLM training

日本語コーパスを MeCab で NIDA_FICTION へ変換し、その JSONL から小型 seq2seq Transformer を Colab GPU で学習します。

この notebook だけで、データ生成 → 10件確認 → tokenizer 学習 → モデル学習 → 保存 → 推論まで通します。

In [ ]:
!pip -q install mecab-python3 unidic-lite datasets sentencepiece

## 1. 設定

In [ ]:
from pathlib import Path
import json, math, random, re
import MeCab
import sentencepiece as spm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

OUTPUT_PATH = Path('/content/kucholm_nida.jsonl')
WORK_DIR = Path('/content/kucholm_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
MAX_ROWS = 100_000
USE_LOCAL_TEXT = False
LOCAL_TEXT_PATH = Path('/content/corpus.txt')
DATASET_NAME = 'range3/cc100-ja'
DATASET_SPLIT = 'train'
TEXT_COLUMN = 'text'
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type != 'cuda':
    print('Runtime > Change runtime type > GPU を推奨')

tagger = MeCab.Tagger()

## 2. コーパス読み込み

In [ ]:
if USE_LOCAL_TEXT:
    corpus = (line.strip() for line in LOCAL_TEXT_PATH.open(encoding='utf-8'))
else:
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    corpus = (str(row[TEXT_COLUMN]).strip() for row in dataset)

## 3. NIDA_FICTION 変換

In [ ]:
URL_RE = re.compile(r'https?://|www\.|```|`[^`]+`')
END_PUNCT = {'。', '！', '!', '？', '?'}

def parse_tokens(text):
    node = tagger.parseToNode(text)
    tokens = []
    while node:
        if node.surface:
            features = node.feature.split(',')
            tokens.append({
                'surface': node.surface,
                'pos': features[0] if len(features) > 0 else '',
                'ctype': features[4] if len(features) > 4 else '*',
                'lemma': features[7] if len(features) > 7 else '*',
                'orth_base': features[10] if len(features) > 10 else '*',
            })
        node = node.next
    return tokens

def dictionary_form(token):
    for key in ('orth_base', 'lemma'):
        value = token.get(key, '*')
        if value not in {'', '*'} and re.search(r'[ぁ-ん一-龯]', value):
            return value
    return token['surface']

def is_ichidan(token, base):
    ctype = token.get('ctype', '')
    if '下一段' in ctype or '上一段' in ctype or '一段' in ctype:
        return True
    return base.endswith('る') and token.get('surface', '') == base[:-1]

def ta_form(base, token):
    if base == '行く': return '行った'
    if base == '来る': return '来た'
    if base == 'する': return 'した'
    if is_ichidan(token, base): return base[:-1] + 'た' if base.endswith('る') else base + 'た'
    if base.endswith(('う', 'つ', 'る')): return base[:-1] + 'った'
    if base.endswith(('む', 'ぶ', 'ぬ')): return base[:-1] + 'んだ'
    if base.endswith('く'): return base[:-1] + 'いた'
    if base.endswith('ぐ'): return base[:-1] + 'いだ'
    if base.endswith('す'): return base[:-1] + 'した'
    return base + 'た'

def nai_form(base, token):
    if base == 'する': return 'しない'
    if base == '来る': return '来ない'
    if is_ichidan(token, base): return base[:-1] + 'ない' if base.endswith('る') else base + 'ない'
    if base.endswith('う'): return base[:-1] + 'わない'
    mapping = {'く':'か','ぐ':'が','す':'さ','つ':'た','ぬ':'な','ぶ':'ば','む':'ま','る':'ら'}
    last = base[-1:]
    return base[:-1] + mapping[last] + 'ない' if last in mapping else base + 'ない'

def find_last_verb(tokens, before_index):
    for index in range(before_index - 1, -1, -1):
        if tokens[index]['pos'] == '動詞':
            return index, tokens[index]
    return None, None

def soft_ending(original, is_question=False):
    if is_question: return 'ニカ'
    if re.search(r'(ね|よ|な|かな|かも|けど|けどね)$', original): return 'ニダ'
    if re.search(r'(ない|ません|難しい|心配|残念|大丈夫)$', original): return 'ニダね'
    return 'ニダよ'

def soften_surface(text):
    for pattern, replacement in [
        (r'ということです$', 'ってこと'),
        (r'ということでした$', 'ってことだった'),
        (r'のであります$', 'んだ'),
        (r'であります$', 'なんだ'),
        (r'なのです$', 'なんだ'),
        (r'のです$', 'んだ'),
        (r'でしょう$', 'だろう'),
        (r'ではありません$', 'じゃない'),
        (r'ではないです$', 'じゃない'),
        (r'ではない$', 'じゃない'),
    ]:
        text = re.sub(pattern, replacement, text)
    return text

def to_nida(text):
    text = text.strip()
    if not text or URL_RE.search(text): return None
    punctuation = text[-1] if text[-1:] in END_PUNCT else ''
    body = soften_surface(text[:-1] if punctuation else text)
    tokens = parse_tokens(body)
    if not tokens: return None
    is_question = punctuation in {'？', '?'}
    if tokens and tokens[-1]['surface'] == 'か':
        tokens.pop()
        is_question = True
    surfaces = [token['surface'] for token in tokens]
    ending = soft_ending(body, is_question)
    tail = punctuation or ('？' if is_question else '')
    for suffix, mode in [
        (['ませ', 'ん', 'でし', 'た'], 'negative_past'),
        (['ませ', 'ん'], 'negative'),
        (['まし', 'た'], 'past'),
        (['ます'], 'present'),
    ]:
        if len(surfaces) < len(suffix) or surfaces[-len(suffix):] != suffix: continue
        verb_index, verb = find_last_verb(tokens, len(tokens) - len(suffix))
        if verb is None: continue
        base = dictionary_form(verb)
        prefix = ''.join(token['surface'] for token in tokens[:verb_index])
        if mode == 'present': replacement = base
        elif mode == 'past': replacement = ta_form(base, verb)
        else:
            negative = nai_form(base, verb)
            replacement = negative if mode == 'negative' else negative[:-2] + 'なかった'
        return prefix + replacement + ending + tail
    if surfaces[-2:] == ['でし', 'た']:
        return ''.join(surfaces[:-2]) + 'だった' + ending + tail
    if surfaces[-1:] == ['です']:
        return ''.join(surfaces[:-1]) + ending + tail
    joined = ''.join(surfaces)
    if joined.endswith(('ね', 'よ', 'な')):
        particle = joined[-1]
        return joined[:-1] + 'ニダ' + particle + tail
    return joined + ending + tail

## 4. JSONL生成

In [ ]:
written = 0
with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for source in corpus:
        source = source.strip()
        if not source or len(source) < 2 or len(source) > 256: continue
        target = to_nida(source)
        if not target or target == source: continue
        output.write(json.dumps({'style': 'NIDA_FICTION', 'source': source, 'target': target}, ensure_ascii=False) + '\n')
        written += 1
        if written >= MAX_ROWS: break
print('written:', written)
print('size MB:', OUTPUT_PATH.stat().st_size / 1024 / 1024)

## 5. 生成サンプルを10件確認

In [ ]:
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10: break
        item = json.loads(line)
        print(f"{i + 1:02d}. {item['source']} -> {item['target']}")

## 6. 学習データ読込

45MB程度なら Colab RAM に読み込めます。入力には `<NIDA_FICTION>` を付けます。

In [ ]:
rows = []
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        rows.append((f"<NIDA_FICTION> {item['source']}", item['target']))
random.shuffle(rows)
split = int(len(rows) * 0.98)
train_rows = rows[:split]
val_rows = rows[split:]
print('train:', len(train_rows), 'val:', len(val_rows))

## 7. SentencePiece tokenizer

In [ ]:
VOCAB_SIZE = 8000
spm_corpus = WORK_DIR / 'spm_corpus.txt'
with spm_corpus.open('w', encoding='utf-8') as f:
    for source, target in rows:
        f.write(source + '\n' + target + '\n')
spm.SentencePieceTrainer.train(
    input=str(spm_corpus),
    model_prefix=str(WORK_DIR / 'kucholm_spm'),
    vocab_size=VOCAB_SIZE,
    model_type='bpe',
    character_coverage=0.9995,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
)
sp = spm.SentencePieceProcessor(model_file=str(WORK_DIR / 'kucholm_spm.model'))
print('vocab:', sp.vocab_size())

## 8. Dataset / DataLoader

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 64 if device.type == 'cuda' else 8
PAD_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.bos_id(), sp.eos_id()

def encode(text):
    ids = sp.encode(text, out_type=int)[:MAX_LEN - 2]
    return [BOS_ID, *ids, EOS_ID]

class PairDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        source, target = self.items[i]
        return torch.tensor(encode(source)), torch.tensor(encode(target))

def collate(batch):
    src, tgt = zip(*batch)
    return (
        nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=PAD_ID),
        nn.utils.rnn.pad_sequence(tgt, batch_first=True, padding_value=PAD_ID),
    )

train_loader = DataLoader(PairDataset(train_rows), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=2, pin_memory=device.type == 'cuda')
val_loader = DataLoader(PairDataset(val_rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=2, pin_memory=device.type == 'cuda')

## 9. KuchoLM 約20M

In [ ]:
D_MODEL = 256
NHEAD = 8
ENC_LAYERS = 3
DEC_LAYERS = 3
FFN = 768
DROPOUT = 0.1

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class KuchoLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(sp.vocab_size(), D_MODEL, padding_idx=PAD_ID)
        self.pos = PositionalEncoding(D_MODEL)
        self.transformer = nn.Transformer(
            d_model=D_MODEL, nhead=NHEAD,
            num_encoder_layers=ENC_LAYERS, num_decoder_layers=DEC_LAYERS,
            dim_feedforward=FFN, dropout=DROPOUT, batch_first=True,
        )
        self.head = nn.Linear(D_MODEL, sp.vocab_size(), bias=False)
        self.head.weight = self.embedding.weight
    def forward(self, src, tgt):
        src_pad = src.eq(PAD_ID)
        tgt_pad = tgt.eq(PAD_ID)
        mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1), device=tgt.device)
        src_emb = self.pos(self.embedding(src) * math.sqrt(D_MODEL))
        tgt_emb = self.pos(self.embedding(tgt) * math.sqrt(D_MODEL))
        hidden = self.transformer(src_emb, tgt_emb, tgt_mask=mask, src_key_padding_mask=src_pad, tgt_key_padding_mask=tgt_pad, memory_key_padding_mask=src_pad)
        return self.head(hidden)

model = KuchoLM().to(device)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params')

## 10. 学習

In [ ]:
EPOCHS = 3
LR = 3e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for step, (src, tgt) in enumerate(train_loader, 1):
        src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            logits = model(src, tgt[:, :-1])
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        if step % 100 == 0:
            print(f'epoch {epoch} step {step}/{len(train_loader)} loss {train_loss / step:.4f}')

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                logits = model(src, tgt[:, :-1])
                val_loss += criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1)).item()
    print(f'epoch {epoch}: train={train_loss / len(train_loader):.4f} val={val_loss / max(1, len(val_loader)):.4f}')
    torch.save({'model': model.state_dict(), 'epoch': epoch}, WORK_DIR / f'kucholm_epoch{epoch}.pt')

## 11. 最終モデル保存

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'vocab_size': sp.vocab_size(), 'max_len': MAX_LEN, 'd_model': D_MODEL,
        'nhead': NHEAD, 'encoder_layers': ENC_LAYERS, 'decoder_layers': DEC_LAYERS, 'ffn': FFN,
    },
}, WORK_DIR / 'KuchoLM-NIDA.pt')
print(WORK_DIR / 'KuchoLM-NIDA.pt')
print(WORK_DIR / 'kucholm_spm.model')

## 12. 推論

In [ ]:
@torch.no_grad()
def convert(text, max_new_tokens=128):
    model.eval()
    src = torch.tensor([encode(f'<NIDA_FICTION> {text}')], device=device)
    generated = torch.tensor([[BOS_ID]], device=device)
    for _ in range(max_new_tokens):
        next_id = model(src, generated)[:, -1].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_id], dim=1)
        if next_id.item() == EOS_ID: break
    ids = generated[0].tolist()[1:]
    if EOS_ID in ids: ids = ids[:ids.index(EOS_ID)]
    return sp.decode(ids)

for text in ['今日は学校です。', '魚を食べました。', 'この方法はとても便利です。', '明日は雨かもしれません。']:
    print(text, '->', convert(text))